# Notebook 3: Frameworks Éticos y Toma de Decisiones

## Objetivos
- Implementar principios éticos fundamentales en agentes de IA
- Desarrollar sistemas de filtrado de contenido
- Crear frameworks de decisión ética
- Establecer mecanismos de advertencia para temas sensibles
- Aplicar consideraciones éticas al agente existente

## Configuración del Agente

In [ ]:
import os
import wikipedia
from langchain_openai import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM
try:
    llm = ChatOpenAI(
        model="gpt-4o",
        openai_api_base=os.environ.get("GITHUB_BASE_URL"),
        openai_api_key=os.environ.get("GITHUB_TOKEN"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor
from langsmith import Client

@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Útil para obtener información sobre personas, lugares o conceptos."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Ocurrió un error: {e}"

tools = [get_wikipedia_summary]

client = Client(None)
prompt = client.pull_prompt("hwchase17/openai-tools-agent")

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ Agente y herramientas listos.")

## 1. Principios Éticos Fundamentales

Los cuatro principios bioéticos fundamentales aplicados a la IA:

In [ ]:
class EthicalPrinciples:
    """
    Framework de principios éticos fundamentales para IA.
    Basado en los cuatro principios bioéticos: Beneficencia, No Maleficencia, Autonomía y Justicia.
    """
    
    def __init__(self):
        self.principles = {
            "beneficence": {
                "description": "Actuar para el bien común y maximizar beneficios",
                "guidelines": [
                    "Priorizar el bienestar de usuarios y sociedad",
                    "Maximizar beneficios positivos",
                    "Promover el uso responsable de la IA",
                    "Considerar impactos a largo plazo"
                ]
            },
            "non_maleficence": {
                "description": "No hacer daño - Prevenir daños",
                "guidelines": [
                    "Evitar generación de contenido dañino",
                    "Prevenir discriminación y sesgos",
                    "Proteger privacidad y datos sensibles",
                    "Minimizar riesgos de seguridad"
                ]
            },
            "autonomy": {
                "description": "Respetar la agencia y decisión humana",
                "guidelines": [
                    "Mantener control humano en decisiones críticas",
                    "Proporcionar transparencia en decisiones",
                    "Permitir opt-out y consentimiento informado",
                    "Evitar manipulación psicológica"
                ]
            },
            "justice": {
                "description": "Trato equitativo y fair",
                "guidelines": [
                    "Garantizar acceso equitativo a la tecnología",
                    "Evitar discriminación en resultados",
                    "Considerar distribuciones justas de beneficios",
                    "Promover inclusión y diversidad"
                ]
            }
        }
    
    def evaluate_action(self, action_description):
        """
        Evalúa una acción según los principios éticos.
        
        Args:
            action_description: Descripción de la acción a evaluar
        
        Returns:
            dict con evaluación por principio
        """
        evaluation = {}
        for principle, info in self.principles.items():
            # En una implementación real, esto usaría NLP para analizar
            # Aquí simulamos la evaluación
            evaluation[principle] = {
                "compliance": "pending_analysis",
                "guidelines": info["guidelines"]
            }
        
        return evaluation
    
    def get_principle_info(self, principle):
        """
        Obtiene información detallada de un principio.
        
        Args:
            principle: Nombre del principio
        
        Returns:
            dict con información del principio
        """
        return self.principles.get(principle, None)

# Crear instancia de principios éticos
ethical_principles = EthicalPrinciples()

print("🏛️ Principios Éticos Fundamentales:")
for principle, info in ethical_principles.principles.items():
    print(f"\n{principle.replace('_', ' ').title()}:")
    print(f"  {info['description']}")
    print("  Directrices:")
    for guideline in info['guidelines']:
        print(f"    • {guideline}")

## 2. Framework Ético Completo

Implementamos un framework completo de decisión ética:

In [ ]:
import re

class EthicalFramework:
    """
    Framework ético completo para agentes de IA.
    """
    
    def __init__(self):
        # Temas prohibidos (no se deben responder)
        self.prohibited_topics = [
            "violence", "hate_speech", "illegal_activities",
            "self_harm", "private_information", "manipulation",
            "terrorism", "extremism", "harassment",
            "child_exploitation", "weapons", "drugs_manufacturing"
        ]
        
        # Temas sensibles (requieren advertencias)
        self.sensitive_topics = [
            "medical_advice", "legal_advice", "financial_advice",
            "political_content", "religious_content", "mental_health"
        ]
        
        # Palabras clave para detección
        self.prohibited_keywords = [
            "matar", "asesinar", "violencia", "odio", "racismo",
            "discriminación", "ilegal", "delito", "suicidio",
            "autolesión", "hackear", "piratear", "terrorismo"
        ]
        
        self.sensitive_keywords = [
            "médico", "legal", "financiero", "político",
            "religioso", "salud mental", "diagnóstico",
            "tratamiento", "inversión", "demanda"
        ]
    
    def contains_topic(self, text, topic):
        """
        Verifica si el texto contiene un tema específico.
        
        Args:
            text: Texto a analizar
            topic: Tema a buscar
        
        Returns:
            bool - True si contiene el tema
        """
        # Búsqueda simple por palabras clave relacionadas
        topic_variations = {
            "violence": ["violencia", "agresión", "golpear", "herir"],
            "hate_speech": ["odio", "racismo", "discriminación", "xenofobia"],
            "illegal_activities": ["ilegal", "delito", "crimen", "fraude"],
            "self_harm": ["suicidio", "autolesión", "matarme", "herirme"],
            "private_information": ["privado", "confidencial", "secreto", "personal"],
            "medical_advice": ["médico", "diagnóstico", "tratamiento", "receta"],
            "legal_advice": ["legal", "abogado", "demanda", "juicio"],
            "financial_advice": ["financiero", "inversión", "dinero", "acciones"]
        }
        
        variations = topic_variations.get(topic, [topic.replace("_", " ")])
        text_lower = text.lower()
        
        for variation in variations:
            if variation in text_lower:
                return True
        
        return False
    
    def ethical_check(self, query, response=None):
        """
        Realiza verificación ética completa de query y response.
        
        Args:
            query: Query del usuario
            response: Response del agente (opcional)
        
        Returns:
            dict con resultado de verificación ética
        """
        result = {
            "allowed": True,
            "warning": None,
            "prohibited_reason": None,
            "detected_topics": []
        }
        
        # Verificar temas prohibidos en query
        for topic in self.prohibited_topics:
            if self.contains_topic(query, topic):
                result["allowed"] = False
                result["prohibited_reason"] = f"Topic '{topic}' is prohibited"
                result["detected_topics"].append(topic)
                return result
        
        # Verificar temas prohibidos en response (si se proporciona)
        if response:
            for topic in self.prohibited_topics:
                if self.contains_topic(response, topic):
                    result["allowed"] = False
                    result["prohibited_reason"] = f"Response contains prohibited topic '{topic}'"
                    result["detected_topics"].append(topic)
                    return result
        
        # Verificar temas sensibles en query
        for topic in self.sensitive_topics:
            if self.contains_topic(query, topic):
                result["warning"] = f"⚠️ Warning: Seek professional {topic.replace('_', ' ')} advice"
                result["detected_topics"].append(topic)
        
        return result
    
    def get_ethical_guidance(self, query):
        """
        Proporciona guía ética para una query específica.
        
        Args:
            query: Query del usuario
        
        Returns:
            str con guía ética
        """
        ethical_result = self.ethical_check(query)
        
        if not ethical_result["allowed"]:
            return f"❌ No puedo responder a esta solicitud: {ethical_result['prohibited_reason']}"
        
        if ethical_result["warning"]:
            return f"⚠️ {ethical_result['warning']}. Puedo proporcionar información general, pero no consejos profesionales."
        
        return "✅ Esta consulta es apropiada para responder."

# Crear instancia del framework ético
ethical_framework = EthicalFramework()

print("✅ Framework Ético configurado.")
print(f"Temas prohibidos: {len(ethical_framework.prohibited_topics)}")
print(f"Temas sensibles: {len(ethical_framework.sensitive_topics)}")

## 3. Sistema de Filtrado de Contenido

Implementamos un sistema avanzado de filtrado de contenido:

In [ ]:
class ContentFilter:
    """
    Sistema de filtrado de contenido para agentes de IA.
    """
    
    def __init__(self):
        self.ethical_framework = EthicalFramework()
        self.filter_stats = {
            "total_queries": 0,
            "allowed": 0,
            "blocked": 0,
            "warning": 0
        }
    
    def filter_query(self, query):
        """
        Filtra una query según criterios éticos.
        
        Args:
            query: Query del usuario
        
        Returns:
            dict con resultado del filtrado
        """
        self.filter_stats["total_queries"] += 1
        
        ethical_result = self.ethical_framework.ethical_check(query)
        
        if not ethical_result["allowed"]:
            self.filter_stats["blocked"] += 1
            return {
                "status": "blocked",
                "message": ethical_result["prohibited_reason"],
                "query": query,
                "detected_topics": ethical_result["detected_topics"]
            }
        
        if ethical_result["warning"]:
            self.filter_stats["warning"] += 1
            return {
                "status": "warning",
                "message": ethical_result["warning"],
                "query": query,
                "detected_topics": ethical_result["detected_topics"]
            }
        
        self.filter_stats["allowed"] += 1
        return {
            "status": "allowed",
            "message": "Query allowed",
            "query": query,
            "detected_topics": []
        }
    
    def filter_response(self, response):
        """
        Filtra una response según criterios éticos.
        
        Args:
            response: Response del agente
        
        Returns:
            dict con resultado del filtrado
        """
        ethical_result = self.ethical_framework.ethical_check("", response)
        
        if not ethical_result["allowed"]:
            return {
                "status": "blocked",
                "message": ethical_result["prohibited_reason"],
                "sanitized_response": self._sanitize_response(response)
            }
        
        return {
            "status": "allowed",
            "message": "Response allowed",
            "response": response
        }
    
    def _sanitize_response(self, response):
        """
        Sanitiza una response removiendo contenido problemático.
        
        Args:
            response: Response a sanitizar
        
        Returns:
            str con response sanitizada
        """
        # Implementación básica de sanitización
        sanitized = response
        for topic in self.ethical_framework.prohibited_topics:
            if self.ethical_framework.contains_topic(response, topic):
                sanitized = sanitized.replace(topic, "[CONTENT_REMOVED]")
        
        return sanitized
    
    def get_stats(self):
        """
        Obtiene estadísticas del filtrado.
        
        Returns:
            dict con estadísticas
        """
        return self.filter_stats
    
    def reset_stats(self):
        """
        Reinicia las estadísticas.
        """
        self.filter_stats = {
            "total_queries": 0,
            "allowed": 0,
            "blocked": 0,
            "warning": 0
        }

# Crear instancia del filtro de contenido
content_filter = ContentFilter()

print("✅ Sistema de Filtrado de Contenido configurado.")

## 4. Pruebas del Framework Ético

Probamos el framework ético con diferentes tipos de consultas:

In [ ]:
# Pruebas del framework ético
test_queries = [
    "¿Qué es la inteligencia artificial?",
    "¿Cómo puedo hackear una cuenta de banco?",
    "¿Cuáles son los síntomas de la depresión?",
    "Necesito un diagnóstico médico para mi dolor de cabeza",
    "¿Cómo puedo hacer un tratamiento legal para mi caso?",
    "¿Cuál es la mejor inversión para mi dinero?",
    "¿Cómo puedo aprender programación?",
    "¿Qué es el aprendizaje automático?"
]

print("🧪 Pruebas del Framework Ético:")
for query in test_queries:
    print(f"\n--- Query: {query} ---")
    guidance = ethical_framework.get_ethical_guidance(query)
    print(f"Guía ética: {guidance}")
    
    filter_result = content_filter.filter_query(query)
    print(f"Estado del filtro: {filter_result['status']}")
    if filter_result['detected_topics']:
        print(f"Temas detectados: {filter_result['detected_topics']}")

## 5. Agente Ético - Integración de Framework

Integramos el framework ético en un wrapper para el agente:

In [ ]:
import time

class EthicalAgentWrapper:
    """
    Wrapper que añade capas éticas al agente de IA.
    """
    
    def __init__(self, agent_executor):
        self.agent_executor = agent_executor
        self.content_filter = ContentFilter()
        self.ethical_framework = EthicalFramework()
        self.decision_log = []
    
    def invoke(self, user_id, query):
        """
        Invoca el agente con verificación ética.
        
        Args:
            user_id: Identificador del usuario
            query: Query del usuario
        
        Returns:
            dict con respuesta y metadatos éticos
        """
        # 1. Filtrar query
        filter_result = self.content_filter.filter_query(query)
        
        if filter_result["status"] == "blocked":
            self._log_decision(user_id, query, "blocked", filter_result["message"])
            return {
                "success": False,
                "ethical_status": "blocked",
                "message": filter_result["message"],
                "response": None,
                "detected_topics": filter_result["detected_topics"]
            }
        
        # 2. Invocar agente
        try:
            response = self.agent_executor.invoke({"input": query})
            raw_output = response['output']
            
            # 3. Filtrar response
            response_filter = self.content_filter.filter_response(raw_output)
            
            if response_filter["status"] == "blocked":
                self._log_decision(user_id, query, "response_blocked", response_filter["message"])
                return {
                    "success": True,
                    "ethical_status": "response_sanitized",
                    "message": response_filter["message"],
                    "response": response_filter["sanitized_response"],
                    "detected_topics": filter_result["detected_topics"]
                }
            
            # 4. Registrar decisión exitosa
            status = "warning" if filter_result["status"] == "warning" else "allowed"
            self._log_decision(user_id, query, status, filter_result.get("message", "Success"))
            
            return {
                "success": True,
                "ethical_status": status,
                "message": filter_result.get("message", "Query processed successfully"),
                "response": raw_output,
                "detected_topics": filter_result["detected_topics"]
            }
            
        except Exception as e:
            self._log_decision(user_id, query, "error", str(e))
            return {
                "success": False,
                "ethical_status": "error",
                "message": f"Error: {e}",
                "response": None,
                "detected_topics": []
            }
    
    def _log_decision(self, user_id, query, status, message):
        """
        Registra la decisión ética en el log.
        """
        log_entry = {
            "timestamp": time.time(),
            "user_id": user_id,
            "query": query[:100],
            "ethical_status": status,
            "message": message
        }
        self.decision_log.append(log_entry)
    
    def get_decision_log(self, user_id=None):
        """
        Obtiene el log de decisiones éticas.
        
        Args:
            user_id: Identificador del usuario (opcional)
        
        Returns:
            list de entradas de log
        """
        if user_id:
            return [entry for entry in self.decision_log if entry['user_id'] == user_id]
        return self.decision_log
    
    def get_ethical_stats(self):
        """
        Obtiene estadísticas éticas.
        
        Returns:
            dict con estadísticas
        """
        return self.content_filter.get_stats()

# Crear el agente ético
ethical_agent = EthicalAgentWrapper(agent_executor)

print("✅ Agente Ético configurado con framework de decisión.")

## 6. Práctica con el Agente Ético

Probamos el agente ético con diferentes escenarios:

In [ ]:
# Pruebas del agente ético
ethical_test_scenarios = [
    {
        "user_id": "user_ethical_1",
        "query": "¿Qué es la inteligencia artificial?",
        "description": "Consulta normal"
    },
    {
        "user_id": "user_ethical_2",
        "query": "¿Cómo puedo hackear una cuenta de correo?",
        "description": "Intento de actividad ilegal"
    },
    {
        "user_id": "user_ethical_3",
        "query": "Tengo dolor de cabeza, ¿qué debo tomar?",
        "description": "Consulta médica (sensible)"
    },
    {
        "user_id": "user_ethical_4",
        "query": "¿Cuáles son los principios de la ética en IA?",
        "description": "Consulta sobre ética"
    },
    {
        "user_id": "user_ethical_5",
        "query": "¿Cómo puedo invertir mi dinero de forma segura?",
        "description": "Consulta financiera (sensible)"
    }
]

print("🧪 Pruebas del Agente Ético:")
for scenario in ethical_test_scenarios:
    print(f"\n--- {scenario['description']} ---")
    print(f"User ID: {scenario['user_id']}")
    print(f"Query: {scenario['query']}")
    
    result = ethical_agent.invoke(scenario['user_id'], scenario['query'])
    print(f"\nEstado ético: {result['ethical_status']}")
    print(f"Mensaje: {result['message']}")
    if result['detected_topics']:
        print(f"Temas detectados: {result['detected_topics']}")
    if result['response']:
        print(f"Response: {result['response'][:200]}..." if len(result['response']) > 200 else f"Response: {result['response']}")

## 7. Análisis de Decisiones Éticas

Revisamos el log de decisiones éticas:

In [ ]:
# Obtener estadísticas éticas
ethical_stats = ethical_agent.get_ethical_stats()

print("📊 Estadísticas Éticas:")
print(f"Total de queries: {ethical_stats['total_queries']}")
print(f"Permitidas: {ethical_stats['allowed']}")
print(f"Bloqueadas: {ethical_stats['blocked']}")
print(f"Con advertencia: {ethical_stats['warning']}")

# Calcular porcentajes
if ethical_stats['total_queries'] > 0:
    allowed_pct = (ethical_stats['allowed'] / ethical_stats['total_queries']) * 100
    blocked_pct = (ethical_stats['blocked'] / ethical_stats['total_queries']) * 100
    warning_pct = (ethical_stats['warning'] / ethical_stats['total_queries']) * 100
    
    print(f"\nPorcentajes:")
    print(f"Permitidas: {allowed_pct:.1f}%")
    print(f"Bloqueadas: {blocked_pct:.1f}%")
    print(f"Con advertencia: {warning_pct:.1f}%")

## 8. Verificación del Log de Decisiones

In [ ]:
# Obtener y mostrar el log de decisiones
decision_log = ethical_agent.get_decision_log()

print("📋 Log de Decisiones Éticas:")
print(f"Total de decisiones: {len(decision_log)}")

for entry in decision_log:
    print(f"\n--- Entry ---")
    print(f"Timestamp: {entry['timestamp']}")
    print(f"User ID: {entry['user_id']}")
    print(f"Query: {entry['query']}")
    print(f"Estado Ético: {entry['ethical_status']}")
    print(f"Mensaje: {entry['message']}")

## 9. Framework de Decisión Ética Avanzado

Implementamos un sistema más sofisticado de decisión ética:

In [ ]:
class AdvancedEthicalDecisionFramework:
    """
    Framework avanzado de decisión ética con ponderación de principios.
    """
    
    def __init__(self):
        self.principles = EthicalPrinciples()
        self.ethical_framework = EthicalFramework()
        
        # Pesos para diferentes principios
        self.principle_weights = {
            "non_maleficence": 0.4,
            "beneficence": 0.3,
            "autonomy": 0.2,
            "justice": 0.1
        }
    
    def evaluate_ethical_score(self, query, response):
        """
        Evalúa el score ético de una query-response pair.
        
        Args:
            query: Query del usuario
            response: Response del agente
        
        Returns:
            dict con score ético y desglose
        """
        # Verificación básica de temas prohibidos
        basic_check = self.ethical_framework.ethical_check(query, response)
        
        if not basic_check["allowed"]:
            return {
                "ethical_score": 0.0,
                "allowed": False,
                "reason": basic_check["prohibited_reason"],
                "principle_scores": {}
            }
        
        # Evaluar según principios (simulado)
        principle_scores = {}
        total_score = 0.0
        
        for principle, weight in self.principle_weights.items():
            # En implementación real, esto usaría NLP para evaluar
            # Aquí simulamos scores basados en la verificación básica
            if basic_check["warning"]:
                score = 0.7
            else:
                score = 1.0
            
            principle_scores[principle] = score
            total_score += score * weight
        
        return {
            "ethical_score": total_score,
            "allowed": True,
            "warning": basic_check.get("warning"),
            "principle_scores": principle_scores,
            "detected_topics": basic_check.get("detected_topics", [])
        }
    
    def get_ethical_recommendation(self, ethical_score, warning=None):
        """
        Proporciona recomendación basada en score ético.
        
        Args:
            ethical_score: Score ético (0-1)
            warning: Advertencia opcional
        
        Returns:
            str con recomendación
        """
        if ethical_score < 0.5:
            return "❌ No recomendado: viola principios éticos fundamentales"
        elif ethical_score < 0.8:
            return "⚠️ Precaución: requiere revisión adicional"
        elif warning:
            return "⚠️ Permitido con advertencia: tema sensible"
        else:
            return "✅ Recomendado: cumple con principios éticos"

# Crear instancia del framework avanzado
advanced_framework = AdvancedEthicalDecisionFramework()

print("✅ Framework Avanzado de Decisión Ética configurado.")
print("Pesos de principios:")
for principle, weight in advanced_framework.principle_weights.items():
    print(f"  {principle}: {weight}")

## 10. Resumen

### Componentes Implementados
- **EthicalPrinciples**: Framework de los cuatro principios bioéticos fundamentales
- **EthicalFramework**: Sistema completo de verificación ética con temas prohibidos y sensibles
- **ContentFilter**: Sistema de filtrado de contenido con estadísticas
- **EthicalAgentWrapper**: Integración de framework ético en el agente
- **AdvancedEthicalDecisionFramework**: Sistema avanzado con ponderación de principios

### Principios Éticos Aplicados
- **Beneficencia**: Actuar para el bien común
- **No Maleficencia**: Prevenir daños
- **Autonomía**: Respetar la agencia humana
- **Justicia**: Trato equitativo y fair

### Funcionalidades Clave
- **Detección de temas prohibidos**: Bloqueo automático de contenido inapropiado
- **Advertencias para temas sensibles**: Alertas para contenido que requiere precaución
- **Logging de decisiones**: Trazabilidad de todas las decisiones éticas
- **Estadísticas de filtrado**: Monitoreo de effectiveness del sistema
- **Score ético**: Evaluación cuantitativa del cumplimiento ético

### Próximos Pasos
- Notebook 4: Protección contra ataques específicos (prompt injection, adversarial attacks)
- Notebook 5: Governance, compliance y monitoring avanzado

### Mejoras Futuras
- Implementar NLP avanzado para detección más precisa
- Agregar aprendizaje automático para adaptación continua
- Implementar sistema de apelación para decisiones bloqueadas
- Agregar evaluación de sesgos en respuestas